In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pathlib import Path

project_root = Path.cwd().parents[1]

def load_bronze_transaction_df():
    bronze_dt_path = (
        project_root
        / "data"
        / "bronze"
    )
    return spark.read.format("delta").load(str(bronze_dt_path))

CATALOG = "workspace"
SCHEMA = "fable_data"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
from functools import reduce

def convert_column_names(bronze_df):
    return (
        bronze_df
        .withColumnRenamed('record_id', 'RECORD_ID')
        .withColumnRenamed('customer_type', 'CUSTOMER_TYPE')
        .withColumnRenamed('transaction_date', 'TRANSACTION_DATE')
        .withColumnRenamed('posting_date', 'POSTING_DATE')
        .withColumnRenamed('transaction_amount', 'TRANSACTION_AMOUNT')
        .withColumnRenamed('Txn_Description', 'ORIGINAL_DESCRIPTION_TEXT')
        .withColumnRenamed('country', 'COUNTRY_CODE')
        .withColumnRenamed('Age_Band', 'AGE_BAND')
        .withColumnRenamed('Transaction_id', 'TRANSACTION_ID')
        .withColumnRenamed('customer_key', 'CUSTOMER_KEY')
        .withColumnRenamed('customer_postcode', 'POSTCODE')
        .withColumnRenamed('Notes', 'NOTES')
    )

def convert_date_strings_to_dates(df):
    return (
        df
        .withColumn("POSTING_DATE", F.to_date(F.col("POSTING_DATE"), "dd/MM/yyyy"))
        .withColumn("TRANSACTION_DATE", F.to_date(F.col("TRANSACTION_DATE"), "dd/MM/yyyy"))
    )

def drop_malformed_rows(df):
    #Drops rows where record_id contains a non-numeric value and every other column is null or empty string
    record_id = F.trim(F.col("RECORD_ID").cast("string"))

    non_numeric_record_id = (
        record_id.isNotNull()
        & (record_id != "")
        & ~record_id.rlike(r"^\d+$")
    )

    other_columns = [
        column_name
        for column_name in df.columns
        if column_name not in ("RECORD_ID", "FILE_DATE", "INGESTED_AT", "NOTES")
    ]

    all_other_values_are_empty = reduce(
        lambda condition, column_name: condition
        & (
            F.col(column_name).isNull()
            | (
                F.trim(F.col(column_name).cast("string"))
                == ""
            )
        ),
        other_columns,
        F.lit(True),
    )

    malformed_row = (
        non_numeric_record_id
        & all_other_values_are_empty
    )

    return df.filter(~malformed_row)

def cast_numeric_strings_numbers(df):
    return df.withColumn(
        "RECORD_ID",
        F.expr("try_cast(RECORD_ID as int)")
    ).withColumn(
        "TRANSACTION_AMOUNT",
        F.expr("try_cast(TRANSACTION_AMOUNT as decimal(10,2))")
        )


VALID_AGE_BANDS = [
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    ">=70",
]


def clean_age_bands(df):
    age_band = F.trim(F.col("AGE_BAND"))
    age_band_without_first_character = F.regexp_replace(
        age_band,
        r"^.",
        "",
    )

    corrected_age_band = (
        F.when(
            age_band.isin(VALID_AGE_BANDS),
            age_band,
        )
        .when(
            age_band_without_first_character.isin(VALID_AGE_BANDS),
            age_band_without_first_character,
        )
    )

    return (
        df
        .withColumn("AGE_BAND", corrected_age_band)
        .filter(F.col("AGE_BAND").isNotNull())
    )



In [0]:
def create_customer_type(df):
    return df.withColumn(
        "CUSTOMER_TYPE",
        F.when(
            (F.trim(F.col("AGE_BAND")) == ">=70")
            | (
                F.lower(F.trim(F.col("gender")))
                == "unknown"
            ),
            F.lit("Unspecified")
        ).otherwise(F.col("CUSTOMER_TYPE"))
    )
    

def create_posting_date(df):
    return df.withColumn(
        "POSTING_DATE",
        F.when(
            (F.col("POSTING_DATE") < F.col("TRANSACTION_DATE")),
            F.col("TRANSACTION_DATE")
        ).otherwise(F.col("POSTING_DATE"))
    )

def add_formatted_description_text(df):
    cleaned_description = F.trim(
        F.replace(
            F.col("ORIGINAL_DESCRIPTION_TEXT"),
            F.lit("#####"),
            F.lit("")
        )
    )

    trailing_word = F.upper(
        F.substring_index(cleaned_description, " ", -1)
    )

    country_code = F.upper(
        F.trim(F.col("COUNTRY_CODE"))
    )

    return df.withColumn(
        "DESCRIPTION_TEXT",
        F.when(
            trailing_word == country_code,
            F.rtrim(
                F.substr(
                    cleaned_description,
                    F.lit(1),
                    F.length(cleaned_description)
                    - F.length(F.trim(F.col("COUNTRY_CODE")))
                )
            )
        ).otherwise(cleaned_description)
    )

In [0]:
EU_COUNTRY_CODES = [
    "AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK",
    "EST", "FIN", "FRA", "DEU", "GRC", "HUN", "IRL",
    "ITA", "LVA", "LTU", "LUX", "MLT", "NLD", "POL",
    "PRT", "ROU", "SVK", "SVN", "ESP", "SWE"
]

def format_country_codes(df):
    df = df.withColumn("COUNTRY_CODE", F.upper(F.trim(F.col("COUNTRY_CODE"))))
    return df

def add_eu_flag(df):
    return df.withColumn(
        "EU_FLAG",
        F.when(
            (F.col("COUNTRY_CODE").isin(EU_COUNTRY_CODES)) |
            ((F.col("COUNTRY_CODE") == "GBR") & (F.col("TRANSACTION_DATE") <= F.to_date(F.lit("2020-1-31"))))
            ,F.lit(True))
        .otherwise(F.lit(False))
    )

In [0]:
def add_gender_column(df):
    """
    Adds the GENDER_CODE column depending on the value of gender, and drops gender
    """
    return df.withColumn(
        "GENDER_CODE",
        F.when(
            F.upper(F.trim(F.col("gender"))).isin("M", "MALE", "MA"),
            "M"
        ).when(
            F.upper(F.trim(F.col("gender"))).isin("F", "FEMALE", "FE"),
            "F"
        ).otherwise("X")
    ).drop("gender")

In [0]:
UK_POSTCODE_REGEX_PATTERN = (
    r"^[A-Z]{1,2}[0-9][A-Z0-9]?\s*[0-9][A-Z]{2}$"
)

def format_post_code(df):
    original_postcode = F.upper(F.col("POSTCODE"))

    postcode_for_check = F.trim(original_postcode)

    postcode_without_spaces = F.regexp_replace(
        F.trim(original_postcode),
        r"\s+",
        ""
    )

    masked_postcode = F.regexp_replace(
        postcode_without_spaces,
        r"..$",
        "**"
    )

    return df.withColumn(
        "POSTCODE",
        F.when(
            postcode_for_check.rlike(UK_POSTCODE_REGEX_PATTERN),
            masked_postcode
        ).otherwise(original_postcode)
    )


In [0]:
#load delta table from data/bronze
#and preform the transofrmations/cleaning
silver_transactions_df = load_bronze_transaction_df()
silver_transactions_df = convert_column_names(silver_transactions_df)
silver_transactions_df = drop_malformed_rows(silver_transactions_df)
silver_transactions_df = clean_age_bands(silver_transactions_df)
silver_transactions_df = cast_numeric_strings_numbers(silver_transactions_df)
silver_transactions_df = convert_date_strings_to_dates(silver_transactions_df)
silver_transactions_df = create_customer_type(silver_transactions_df)
silver_transactions_df = create_posting_date(silver_transactions_df)
silver_transactions_df = add_formatted_description_text(silver_transactions_df)
silver_transactions_df = add_gender_column(silver_transactions_df)
silver_transactions_df = format_country_codes(silver_transactions_df)
silver_transactions_df = add_eu_flag(silver_transactions_df)
silver_transactions_df = format_post_code(silver_transactions_df)

In [0]:
#save the silver table in the data/silver folder
silver_transactions_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_transactions")